In [0]:
# Import packages and functions needed 
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StructType
from pyspark.sql.types import StringType


# Function that identifies and explodes array and struct fields
def expand_arrays(df):
    array_columns = []
    
    # search and select the array columns
    for field in df.schema.fields:
        if isinstance(field.dataType, ArrayType):
            array_columns.append(field.name)

    # loop the array columns
    for col_name in array_columns:
        # Explode each array column
        df = df.withColumn(f"exploded_{col_name}", F.explode_outer(col_name))
        
        # Verify whether the data type of the array is a struct
        element_type = df.schema[f"exploded_{col_name}"].dataType
        if isinstance(element_type, StructType):
            # If it does, extract the struct subfields, create new fields and delete the original one
            for subfield in element_type.fieldNames():
                df = df.withColumn(f"{col_name}_{subfield}", F.col(f"exploded_{col_name}.{subfield}"))
                df = df.drop(col_name)
        else:
            # if the data type is not StructType (ie, StringType o DoubleType), replace the original column with the exploded one
            df = df.withColumn(col_name, F.col(f"exploded_{col_name}"))
        
        # delete the temporal exploded field
        df = df.drop(f"exploded_{col_name}")
    
    return df


# Función para separar el campo usando '~' y ',' como delimitadores
def split_multiple_delimiters(df, input_col, output_col):
    # Usamos regexp_replace para normalizar los delimitadores a uno solo (por ejemplo ',')
    normalized_col = F.regexp_replace(input_col, '[~,]+', ',')
    # Hacemos split del resultado normalizado
    return df.withColumn(output_col, F.split(normalized_col, ','))


# Definir la función para capitalizar la primera letra, limpiar espacios y reemplazar '_'
def clean_text(df, input_col, output_col):
    return df.withColumn(
        output_col,
        F.trim(F.regexp_replace(F.col(input_col), '_', ' '))
    )

# funcion para extraer el contenido entre corchetes
def extract_string_content(df, input_col, output_col):
    return df.withColumn(
        output_col,
        F.when(
            F.col(input_col).rlike(r'\[.*?\]'),  # Si contiene corchetes
            F.regexp_replace(  # Remover las comillas dobles después de extraer el contenido
                F.regexp_extract(F.col(input_col), r'\[(.*?)\]', 1),
                r'"', ''  # Reemplazar todas las comillas dobles por un string vacío
            )
        ).otherwise(F.col(input_col))  # Si no tiene corchetes, dejar el valor original
    )


In [0]:
# Import tables

################ dim_gdm table #################
dim_gdm = spark.table("crm_reporting.dim_gdm_brand_profile")
# print(dim_gdm.count()) #12769048
# print(len(gdm.columns))


####### FIRST SELECTION OF ATTRIBUTES WITH AT LEAST 1 NON-VALUE
df = dim_gdm
# Count non-null values per column
agg_expr = [F.count(F.when(F.col(c).isNotNull(), c)).alias(c) for c in df.columns]

# Execute the aggregation
non_null_counts = df.agg(*agg_expr).collect()[0]

# Keep the not enterely null columns
non_completely_null_columns = [c for c in df.columns if non_null_counts[c] > 0]
# completely_null_columns = [c for c in df.columns if non_null_counts[c] == 0]

# count of not-enterely null columns
# print(len(non_completely_null_columns)) ## 78
# print(len(completely_null_columns)) ## 117
# print(completely_null_columns)

# select the founded fields
gdm_cols_filt = df.select(non_completely_null_columns).\
  withColumnRenamed("created_dt",'created_date').\
  withColumn("created_date",F.to_date('created_date'))



##### OTHER COLUMNS TO REMOVE
columns_to_remove = ['dept_store_regional','etl_batch_id','mdm_source','sys_created_by','beauty_supply_store','category','product_category']

##### Filter columns that contain the string '_dt' and the ones in columns_to_remove
columns_wt_dt = [col for col in gdm_cols_filt.columns if '_dt' not in col and col not in columns_to_remove]

# count of not-enterely null columns
# print(len(columns_wt_dt)) ## 51
#print(non_completely_null_columns)


# select the list of columns to keep
gdm_cols_filt = gdm_cols_filt.select(columns_wt_dt)
gdm_cols_filt.createOrReplaceTempView("gdm_cols_filt_vw")

In [0]:
# call the expand array function twice
expld_gdm_1 = expand_arrays(gdm_cols_filt) # explode first array levels
expld_gdm_1 = expand_arrays(expld_gdm_1) # explode second array levels

# print(expld_gdm_1.count()) # 433098
# print(len(expld_gdm.columns)) ## 81
# display(expld_gdm.limit(1))
expld_gdm_1.createOrReplaceTempView("expld_gdm_1_vw")
# print(expld_gdm_1.count()) # 13155657

In [0]:
# SECOND SELECTION OF ATTRIBUTES WITH AT LEAST 1 NON-VALUE

df = expld_gdm_1

# Count non-null values per column
agg_expr = [F.count(F.when(F.col(c).isNotNull(), c)).alias(c) for c in df.columns]

# Execute the aggregation
non_null_counts = df.agg(*agg_expr).collect()[0]

# Keep the not enterely null columns
non_completely_null_columns = [c for c in df.columns if non_null_counts[c] > 0]

# count of not-enterely null columns
# print(len(non_completely_null_columns)) ## 47

# select the list of columns to keep
def_gdm_v1 = df.select(non_completely_null_columns).\
  drop('analysis_exact_age','analysis_calculated_age')

# limpiamos valores con la estructura: Concern_type="texture uniformity";zone=["whole face", "cheek"]
# def_gdm_v1 = extract_string_content(def_gdm_v1, "analysis_analysis_concern_zone", "analysis_analysis_concern_zone")
def_gdm_v1.createOrReplaceTempView("def_gdm_v1_vw")

In [0]:
# # Apply split function in the required fields
# def_gdm = split_multiple_delimiters(def_gdm_v1, "concern_improvement_goal_concern", "concern_improvement_goal_concern")
# def_gdm = split_multiple_delimiters(def_gdm, "analysis_analysis_concern_concern", "analysis_analysis_concern_concern")
# def_gdm = split_multiple_delimiters(def_gdm, "analysis_analysis_concern_zone", "analysis_analysis_concern_zone")
# def_gdm = split_multiple_delimiters(def_gdm, "channel_preference_product_purchase_channel", "channel_preference_product_purchase_channel")
# def_gdm = split_multiple_delimiters(def_gdm, "hair_routine_heating_heating_tool", "hair_routine_heating_heating_tool")
# def_gdm = split_multiple_delimiters(def_gdm, "analysis_analysis_type", "analysis_analysis_type")
# def_gdm = split_multiple_delimiters(def_gdm, "analysis_analysis_concern_concern_type", "analysis_analysis_concern_concern_type")
# def_gdm = split_multiple_delimiters(def_gdm, "goal", "goal")
# def_gdm = split_multiple_delimiters(def_gdm, "makeup_look", "makeup_look")
# def_gdm = split_multiple_delimiters(def_gdm, "makeup_priority", "makeup_priority")
# def_gdm = split_multiple_delimiters(def_gdm, "makeup_motivation", "makeup_motivation")
# def_gdm = split_multiple_delimiters(def_gdm, "scalp_type", "scalp_type")
# def_gdm = split_multiple_delimiters(def_gdm, "external_factor", "external_factor")
# def_gdm = split_multiple_delimiters(def_gdm, "fragrance_priority", "fragrance_priority")
# def_gdm = split_multiple_delimiters(def_gdm, "skin_type_skin_type", "skin_type_skin_type")
# def_gdm = split_multiple_delimiters(def_gdm, "hair_type", "hair_type")
# def_gdm = split_multiple_delimiters(def_gdm, "hair_color", "hair_color")
# def_gdm = split_multiple_delimiters(def_gdm, "natural_hair_color", "natural_hair_color")
# def_gdm = split_multiple_delimiters(def_gdm, "skin_tone", "skin_tone")

# def_gdm = expand_arrays(def_gdm) # explode first array levels

# # Last view before unpivot
# def_gdm.createOrReplaceTempView("def_gdm_vw")

In [0]:
###  UNPIVOT DEF_GDM
# 1. Identificar dinámicamente las columnas de atributos
non_attributes = ["brand_code","brand_country",'created_date','brand_mdm_id']
attributes = [c for c in def_gdm_v1.columns if c not in non_attributes] 

# 2. Crear la expresión para el unpivot con stack()
num_atributos = len(attributes)
stack_expr = ", ".join(
    [f"'{col}', {col}" for col in attributes]
)

# 3. Realizar el unpivot usando `stack()`
gdm_unpivot = def_gdm_v1.select(
    *non_attributes,
    F.expr(f"stack({num_atributos}, {stack_expr}) as (attribute, value)")).distinct()

# remove white spaces and capitalize values
# gdm_unpivot = clean_text(gdm_unpivot, "value", "value_k")

#remover lineas nulas
# gdm_unpivot = gdm_unpivot.filter(F.col("value").isNotNull())

# print(gdm_unpivot.count()) # 36169590 lineas quitando nulos, 12457382 distinct brand_mdm_id
gdm_unpivot.createOrReplaceTempView("gdm_unpivot_vw")

# tmp = gdm_unpivot.select('value','value_k').distinct()
# display(tmp)

In [0]:
# %sql
# select distinct brand_mdm_id
# from gdm_unpivot_vw
# where attribute = 'hair_length'
#   and value = 'Coarse'

In [0]:
# Filtrar valores que tienen 6 o menos caracteres numericos para quitar hasheados
hashed = gdm_unpivot.filter(F.col("value").rlike(r'(.*[0-9]){6,}')).\
  select('brand_mdm_id').distinct()  #65047 con hasheados
hashed.createOrReplaceTempView("hashed_vw")

query = f"""
select
  a.* 
from gdm_unpivot_vw a
inner join hashed_vw b
  on a.brand_mdm_id = b.brand_mdm_id 
"""

hashed_all_attrib = spark.sql(query)
display(hashed_all_attrib)



# tmp = gdm_unpivot.select('brand_mdm_id').distinct() # 12635107 total

# print(tmp.count())

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecutio

In [0]:
# display(tmp)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
# %sql
# select distinct attribute--attribute_databricks
# -- from prod_latam_catalog.crm_analytics.
# from gdm_unpivot_vw
# order by attribute

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
# %sql
# select count(distinct brand_mdm_id)
# from gdm_unpivot_vw

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
# # JOIN DE LA TABLA DE MAPPING CON LOS VALORES DE LA GDM
# mapping = spark.table('prod_latam_catalog.crm_analytics.mapping_dim_gdm_phase2_p1')
# mapping.createOrReplaceTempView("mapping_vw")

# # Extraer los valores de la columna "attribute" desde los Row objects
# attributes = [row["attribute_databricks"] for row in mapping.select("attribute_databricks").distinct().collect()]
# # print(attributes)

# dim_gdm_unpivot = gdm_unpivot.filter(F.col("attribute").isin(attributes))
# # print(dim_gdm_unpivot.count())

# dim_gdm_unpivot.createOrReplaceTempView("dim_gdm_unpivot_vw")

# # print(result.count()) # 2483860 lines using single-choice attributes (16)
# # display(result)

# query = f"""
# with tmp_mapping_dim_gdm as (
#   select
#     attribute_databricks,
#     value_databricks,
#     Mapping,
#     case when value_databricks is not null then 'mapped' else 'mapped but null' end as fix_flag
#   from mapping_vw
# )
# select distinct
#   a.*,
#   b.Mapping,
#   --case when a.value = b.Mapping then 'mapped' else null end as fix_flag
#   b.fix_flag
# from dim_gdm_unpivot_vw a
# left join tmp_mapping_dim_gdm b 
#   on a.attribute = b.attribute_databricks
#   and a.value = b.value_databricks
# where b.fix_flag is not null
# """

# result = spark.sql(query) # quitar nulos, duplicados y hacer flag
# result.createOrReplaceTempView("result_vw")

# # counts 2525247 # single-choice

# # Para crear una tabla permanente en el catalogo
# result.write.mode('overwrite').option("mergeSchema", "true").format("delta").saveAsTable("prod_latam_catalog.crm_analytics.dim_gdm_brand_profile_mapped_phase2_p1")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
# %sql
# select count(distinct brand_mdm_id)
# from result_vw
# where fix_flag is not null

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
## SE AGRUPAN LOS IDS POR ATRIBUTO Y VALOR
gdm_unpivot_gp = gdm_unpivot.groupBy("attribute",'value').agg(
        F.countDistinct("brand_mdm_id").alias("id_counts"),
        #F.min("created_date").alias("min_date"),
        F.max("created_date").alias("max_date")
    )

# print(gdm_unpivot_gp.count())
gdm_unpivot_gp.createOrReplaceTempView("gdm_unpivot_gp_vw")
# display(tmp)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
data = [
    ('analysis_analysis_clinical_sign_clinical_sign','CLINICAL_SIGN'),
    ('analysis_analysis_clinical_sign_sign_type','SIGN_TYPE'),
    ('analysis_analysis_clinical_sign_zone','ZONE'),
    ('analysis_analysis_concern_concern','CONCERN'),
    ('analysis_analysis_concern_concern_type','CONCERN_TYPE'), 
    ('analysis_analysis_concern_zone','ZONE'),
    ('analysis_analysis_type','ANALYSIS_TYPE'),
    ('channel_preference_product_purchase_channel','PRODUCT_PURCHASE_CHANNEL'),
    ('concern_improvement_goal_concern','CONCERN'),
    ('concern_zone','ZONE'),
    ('desired_color_finish','HAIR_COLOR_FINISH'),
    ('desired_color_permanence','HAIR_COLOR_PERMANENCE'), 
    ('desired_cover','HAIR_COLOR_COVER'), 
    ('fragrance_routine_perfume_moment','PERFUME_MOMENT'),
    ('fragrance_priority','FRAGRANCE_MOTIVATION'),
    ('hair_care_product_used','HAIRCARE_PRODUCT'),
    ('hair_routine_heating_heating_tool','HEATING_TOOL'),
    ('improvement_goal_concern','CONCERN'),
    ('last_hair_color_service','HAIR_COLOR_SERVICE'),
    ('last_hair_style_look','HAIR_STYLE_LOOK'),
    ('last_skincare_medical_treatment','LAST_SKINCARE_PROFESSIONAL_TREATMENT'),
    ('left_eye_color','EYE_COLOR'),
    ('look_occasion_makeup','MAKEUP_LOOK_OCCASION'),
    ('makeup_priority','MAKEUP_EXPECTATION'),
    ('natural_hair_color','HAIR_COLOR'),
    ('right_eye_color','EYE_COLOR'),
    ('skin_sensitivity_skin_sensitivity','SKIN_SENSITIVITY'),
    ('skin_sensitivity_zone','ZONE'),
    ('skin_type_skin_type','SKIN_TYPE'),
    ('desired_color_finish','ZONE'), 
    ('desired_style','HAIR_STYLE'), 
    ('skin_type_zone','ZONE')
]

  # withColumnRenamed("hair_routine_heating_heating_tool","heating_tool").\
  # withColumnRenamed("skin_sensitivity_skin_sensitivity","skin_sensitivity").\
  # withColumnRenamed("fragrance_routine_perfume_moment","perfume_moment").\
  # withColumnRenamed("concern_improvement_goal_concern","improvement_goal_concern").\
  # withColumnRenamed("channel_preference_product_purchase_channel","purchase_channel").\
  # withColumnRenamed("analysis_analysis_concern_concern","analysis_concern").\
  # withColumnRenamed("analysis_analysis_concern_concern_type","concern_type").\
  # withColumnRenamed("analysis_analysis_concern_zone","concern_zone").\
  # withColumnRenamed("analysis_analysis_type","analysis_type").\

# Crear el DataFrame
columns = ["attribute", "attribute_k"]
attributes_mapping = spark.createDataFrame(data, columns)
attributes_mapping.createOrReplaceTempView("attributes_mapping_vw")


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
bmdm_lkp = spark.table("prod_latam_catalog.crm_reporting.lkp_bmdm_reference")
# bmdm_global = spark.table("prod_latam_catalog.crm_analytics.bmdm_mapping_lkp_table")

# some transformations
bmdm_lkp = bmdm_lkp.withColumn('reference_name', F.lower('reference_name')).\
  select('reference_name','reference_value').\
  orderBy('reference_name','reference_value')	
# bmdm_lkp = clean_text(bmdm_lkp, "reference_value", "value_k")

bmdm_lkp.createOrReplaceTempView("bmdm_lkp_vw")
# display(bmdm_lkp.limit(5))

# tmp = bmdm_lkp.filter(F.col("reference_name").isin(['ROUTINE_GOAL','ANALYSIS_TYPE'])).\
#   select('reference_name','value_k','reference_value').distinct()
display(bmdm_lkp)



com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
## CRUCE CON LA LKP
dim_gdm_mapping = spark.sql("""
with merge_1 as (
select
  a.*,
  CASE WHEN b.attribute_k IS NULL THEN a.attribute
    ELSE lower(b.attribute_k) END as attribute_k
from gdm_unpivot_gp_vw a
left join attributes_mapping_vw b
  on a.attribute = b.attribute
)

--merge_2 as (
select 
  a.*,  
  c.reference_value as value_lkp
from merge_1 a
left join bmdm_lkp_vw c
  on a.attribute_k = c.reference_name
  and a.value = c.reference_value
--),
""")

# Filtrar valores que tienen 6 o menos caracteres numericos para quitar hasheados
dim_gdm_mapping = dim_gdm_mapping.filter(~F.col("value").rlike(r'(.*[0-9]){6,}'))

#display(tmp)
# print(dim_gdm_mapping.count()) # 33024 opciones sin split
dim_gdm_mapping.createOrReplaceTempView("dim_gdm_mapping_vw")


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
attr = ['analysis_analysis_concern_concern',	'analysis_analysis_concern_concern_type',	'analysis_analysis_concern_zone',	'analysis_analysis_type',	'channel_preference_product_purchase_channel',	'concern_improvement_goal_concern',	'external_factor',	'fragrance_priority',	'goal',	'hair_color',	'hair_type',	'makeup_look',	'makeup_motivation',	'makeup_priority',	'natural_hair_color',	'scalp_type',	'skin_tone',	'skin_type_skin_type',] #32811 multi-choice

# attr_2 = ['concern_improvement_goal_concern'] # 24779
# attr_2 = ['fragrance_priority',	'goal',	'hair_color',	'hair_type',	'makeup_look',	'makeup_motivation',	'makeup_priority',	'natural_hair_color',	'scalp_type',	'skin_tone',	'skin_type_skin_type','channel_preference_product_purchase_channel','external_factor'] # 6510

tmp = dim_gdm_mapping.filter(F.col('attribute').isin(attr))#.\
  #withColumn('value',F.regexp_replace(F.col('value'), ',', '|')) # para evitar que las (,) reales se confundan con las que se crean de separador al exportar el csv, SOLO para imprimir

# print(tmp.count()) # 33024 valores sin split

# display(tmp)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
# Para crear una tabla permanente en el catalogo
# dim_gdm_mapping.write.mode('overwrite').option("mergeSchema", "true").format("delta").saveAsTable("prod_latam_catalog.crm_analytics.all_values_dim_gdm_brand_profile")

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
# %sql
# select distinct date_of_birth
# from crm_reporting.dim_gdm_brand_profile
# order by date_of_birth

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
# tmp = dim_gdm_mapping.filter(F.col("value").rlike("~|,")).\
#   select('attribute').distinct()

# display(tmp)

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
# gdm_exploded
# tmp = def_gdm_v1.groupBy("desired_color_finish").agg( #skin_sensitivity
#         F.countDistinct("brand_mdm_id").alias("id_counts"),
#         F.max("created_date").alias("max_date")
#     )
# display(tmp)

## gdm original
dim_gdm = spark.table("crm_reporting.dim_gdm_brand_profile")

tmp = dim_gdm.groupBy(F.col("gender")).agg(
        F.countDistinct("brand_mdm_id").alias("id_counts"),
        F.max("created_dt").alias("max_date")
    )
display(tmp) 

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
# gdm_exploded
# tmp = def_gdm_v1.groupBy("desired_color_finish").agg( #skin_sensitivity
#         F.countDistinct("brand_mdm_id").alias("id_counts"),
#         F.max("created_date").alias("max_date")
#     )
# display(tmp)

## gdm original
dim_gdm = spark.table("crm_reporting.dim_gdm_brand_profile")

tmp = dim_gdm.groupBy(F.col("look_occasion_hair")).agg(
        F.countDistinct("brand_mdm_id").alias("id_counts"),
        F.max("created_dt").alias("max_date")
    )
display(tmp) 
# 'To pamper myself, it’s a pleasant personal moment'

look_occasion_hair,id_counts,max_date
null,13077497,2025-03-06T00:46:16Z


In [0]:
## DIFERENCIAR ENTRE VARIABLES
a= '’'
b= "'"

# Convertir a bytes y luego a representación hexadecimal
a_hex = a.encode("utf-8").hex()
b_hex = b.encode("utf-8").hex()

print('iguales?: ',a_hex == b_hex)
print('a=',a_hex,'b=',b_hex)

# 'fragile~Hair loss'
# 'softness~Frizzy'

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:447)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperation$1(UsageLogging.scala:573)
	at com.databricks.logging.UsageLogging.executeThunkAndCaptureResultTags$1(UsageLogging.scala:669)
	at com.databricks.logging.UsageLogging.$anonfun$recordOperationWithResultTags$4(UsageLogging.scala:687)
	at com.databricks.logging.UsageLogging.$anonfun$withAttributionContext$1(UsageLogging.scala:426)
	at scala.util.DynamicVariable.withValue(DynamicVariable.scala:62)
	at com.databricks.logging.AttributionContext$.withValue(AttributionContext.scala:216)
	at com.databricks.logging.UsageLogging.withAttributionContext(UsageLogging.scala:424)
	at com.databricks.logging.Usa

In [0]:
# %sql
# select
#   snapshot_year_month,
#   sum(lifetime_consumer_cnt) as lifetime_consumer_cnt
# from crm_reporting.vw_summary_segment_by_brand
# where brand_country = 'BRA'
#   and brand_code = 'DMC'
#   -- and snapshot_year_month != '202412'
#   and lifetime_all_enriched_profile = 'Y'
# group by snapshot_year_month
# order by snapshot_year_month

snapshot_year_month,lifetime_consumer_cnt
202011,127480
202101,1515195
202102,1560840
202103,1613593
202104,1679017
202105,1728823
202106,1780708
202107,1832004
202108,1865556
202109,1895641


In [0]:
%sql
select 
  snapshot_year_month,
  sum(lifetime_consumer_cnt) as lifetime_consumer_cnt
from crm_reporting.vw_summary_segment_by_brand_cont_enrich_agg
where brand_country = 'BRA'
  and brand_code = 'DMC'
  -- and snapshot_year_month != '202412'
  -- and lifetime_all_enriched_profile = 'Y'
group by snapshot_year_month
order by snapshot_year_month

snapshot_year_month,lifetime_consumer_cnt
202011,2229557
202101,2226631
202102,2285310
202103,2358560
202104,2455266
202105,2524756
202106,2609912
202107,2695605
202108,2763180
202109,2818327


In [0]:
%sql
select
  snapshot_year_month,
  sum(lifetime_consumer_cnt) as lifetime_consumer_cnt
from crm_reporting.vw_summary_segment_by_brand
where brand_country = 'BRA'
  and brand_code = 'DMC'
  -- and snapshot_year_month != '202412'
  -- and lifetime_all_enriched_profile = 'Y'
group by snapshot_year_month
order by snapshot_year_month

snapshot_year_month,lifetime_consumer_cnt
202011,2229557
202101,2226631
202102,2285310
202103,2358560
202104,2455266
202105,2524756
202106,2609912
202107,2695605
202108,2763180
202109,2818327


In [0]:
%sql
select * from prod_latam_catalog.crm_reporting.lkp_bmdm_reference

reference_name,reference_code,reference_value,ref_api_name,ref_schema,concern_type,analysis_type,sign_type,zone,category,product_category,routine_type,sys_created_dt,sys_created_by,etl_batch_id
age,lessthan14,<14,ages,beautyprofileref,null,null,null,null,null,null,null,2021-12-16T07:36:34.557Z,spark_user,10207_2021121614012
age,14-17,14-17,ages,beautyprofileref,null,null,null,null,null,null,null,2021-12-16T07:36:34.557Z,spark_user,10207_2021121614012
age,18-19,18-19,ages,beautyprofileref,null,null,null,null,null,null,null,2021-12-16T07:36:34.557Z,spark_user,10207_2021121614012
age,20-24,20-24,ages,beautyprofileref,null,null,null,null,null,null,null,2021-12-16T07:36:34.557Z,spark_user,10207_2021121614012
age,25-29,25-29,ages,beautyprofileref,null,null,null,null,null,null,null,2021-12-16T07:36:34.557Z,spark_user,10207_2021121614012
age,30-34,30-34,ages,beautyprofileref,null,null,null,null,null,null,null,2021-12-16T07:36:34.557Z,spark_user,10207_2021121614012
age,35-39,35-39,ages,beautyprofileref,null,null,null,null,null,null,null,2021-12-16T07:36:34.557Z,spark_user,10207_2021121614012
age,40-44,40-44,ages,beautyprofileref,null,null,null,null,null,null,null,2021-12-16T07:36:34.557Z,spark_user,10207_2021121614012
age,45-49,45-49,ages,beautyprofileref,null,null,null,null,null,null,null,2021-12-16T07:36:34.557Z,spark_user,10207_2021121614012
age,50-54,50-54,ages,beautyprofileref,null,null,null,null,null,null,null,2021-12-16T07:36:34.557Z,spark_user,10207_2021121614012


In [0]:
# df = spark.table("crm_reporting.dim_gdm_brand_profile")
df = def_gdm_v1

# Filtrar columnas que contienen la palabra 'wholeface' en al menos una fila
columns_with_wholeface = [
    c for c in df.columns if df.filter(F.col(c).contains('wholeface')).count() > 0
]

# Mostrar solo las columnas que contienen 'wholeface'
print(columns_with_wholeface)

['skin_sensitivity_zone', 'skin_type_zone']
